# STAIR4-v2.1 (BCCR) — Kaggle Training, Verification, and Experiment Telemetry

This notebook trains the baseline-preserving **STAIR4-v2.1 BCCR** implementation. It uses the YAML configuration as the single source of defaults and sends only documented v2.1 CLI overrides.


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies Khóa Phiên Bản & Đồng Bộ Mã Nguồn
- Đồng bộ repository `STAIR-Enhanced` một cách an toàn (bảo toàn local changes, tránh `git reset --hard`).
- Cài đặt các gói phụ thuộc với phiên bản đã khóa (`freerec==0.9.7`, `torchdata==0.8.0` (--no-deps), `nvidia-ml-py`, `prettytable>=3.5.0`, `pyyaml>=6.0`, `matplotlib`, `scipy`, `pandas`).
- Kích hoạt cơ chế **FreeRec & TorchData Compatibility Shims** (`models/freerec_compat.py`) giải quyết triệt để vấn đề deprecation của `torchdata.datapipes` trên PyTorch 2.x và Python 3.10-3.14.
- Khởi chạy tiến trình con xác thực (`Subprocess Environment Verification`) để bảo đảm môi trường chạy `main_stair4_v2.py` nạp đầy đủ dependencies trước khi tiến hành thực nghiệm.
- Thiết lập biến môi trường seed-controlled (`CUBLAS_WORKSPACE_CONFIG=:4096:8`, `PYTHONPATH`).
- Ghi nhận thông tin phần cứng GPU và phiên bản runtime để đảm bảo tính tái lập (Reproducibility).

In [ ]:
# Cell 1: Environment, dependency setup, and immutable source manifest
import os, subprocess, sys, torch

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# Clone only when the project is absent.  Do not force-checkout origin/main here:
# it can replace the reviewed v2.1 files mounted or edited in the Kaggle session.
if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
    ], check=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
os.chdir(active_dir)
for path in [active_dir, '/kaggle/working']:
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ['PYTHONPATH'] = f"{active_dir}:{os.environ.get('PYTHONPATH', '')}"
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

print('Installing/checking compatible runtime dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'pynvml'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.8.0'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch-geometric', 'freerec==0.9.7', 'nvidia-ml-py',
    'prettytable>=3.5.0', 'pyyaml>=6.0', 'matplotlib', 'scipy', 'pandas'
], check=False)

for name in list(sys.modules):
    if name.startswith(('models.', 'optimizers.', 'freerec')):
        sys.modules.pop(name, None)

verify = [
    sys.executable, '-c',
    'import sys, torch; import models.freerec_compat; import freerec; '
    'from models.stair4_v2 import STAIR4V2, STAIR4V2Options; '
    'print(f"Python={sys.version.split()[0]}, PyTorch={torch.__version__}, FreeRec={freerec.__version__}")'
]
result = subprocess.run(verify, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(f"Runtime verification failed:\n{result.stderr}")

try:
    git_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    git_sha = 'unknown'

print('=' * 80)
print(f'Runtime : {result.stdout.strip()}')
print(f'Git SHA : {git_sha}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print('Source directory is preserved; no force checkout was performed.')
print('=' * 80)



## Cell 2 📂 Chuẩn bị Dữ liệu An Toàn Dung Lượng (Canonical Data Staging)
Tạo thư mục dữ liệu ghi đơn nhất (`CANONICAL_DATA_ROOT = /kaggle/working/data`) để tránh việc nhân bản dữ liệu nhiều lần gây tràn đĩa:
- Ưu tiên tạo **symlinks** từ `/kaggle/input` sang thư mục đích.
- Kiểm tra dung lượng đĩa còn trống trước khi sao chép (`shutil.disk_usage`).
- Chỉ sao chép các tệp chưa tồn tại hoặc khác kích thước.
- Thứ tự chuẩn hóa: **Amazon Baby**, **Amazon Sports**, **Amazon Electronics**.

In [ ]:
# Cell 2: Chuẩn bị dữ liệu an toàn dung lượng từ Kaggle Input
import os, shutil, glob, json

# Dùng một canonical writable root duy nhất để tránh nhân bản dữ liệu lãng phí disk
CANONICAL_DATA_ROOT = '/kaggle/working/data' if os.path.exists('/kaggle/working') else os.path.abspath('data')
PROCESSED_DIR = os.path.join(CANONICAL_DATA_ROOT, 'Processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Nếu đang ở STAIR-DIR, liên kết thư mục data nội bộ sang canonical root
internal_data = os.path.abspath('data')
if internal_data != os.path.abspath(CANONICAL_DATA_ROOT):
    os.makedirs(internal_data, exist_ok=True)
    internal_proc = os.path.join(internal_data, 'Processed')
    if not os.path.exists(internal_proc):
        try:
            os.symlink(PROCESSED_DIR, internal_proc)
        except Exception:
            pass

TARGET_DATASETS = {
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

def safe_stage_dataset(src_dir, folder_name):
    '''Liên kết dữ liệu an toàn sang CANONICAL_DATA_ROOT với kiểm tra disk space'''
    dst_root = os.path.join(CANONICAL_DATA_ROOT, folder_name)
    dst_proc = os.path.join(PROCESSED_DIR, folder_name)

    for dst in [dst_root, dst_proc]:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isdir(s_item):
                if not os.path.exists(d_item):
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(d_item) or os.path.getsize(d_item) != os.path.getsize(s_item):
                    try:
                        if os.path.exists(d_item):
                            os.remove(d_item)
                        os.symlink(s_item, d_item)
                    except Exception:
                        # Kiểm tra dung lượng đĩa trước khi copy
                        free_gb = shutil.disk_usage(CANONICAL_DATA_ROOT).free / (1024**3)
                        if free_gb < 2.0:
                            raise RuntimeError(f"Dung lượng đĩa quá thấp ({free_gb:.2f} GiB)! Không thể copy {s_item}")
                        shutil.copy2(s_item, d_item)

print("=" * 80)
print("TIẾN TRÌNH DÒ TÌM VÀ LIÊN KẾT DỮ LIỆU TỪ KAGGLE INPUT:")
print("=" * 80)

prepared_data = {}
all_input_paths = glob.glob('/kaggle/input/**/*', recursive=True) if os.path.exists('/kaggle/input') else []
data_manifest = {}

for key, (folder_name, patterns) in TARGET_DATASETS.items():
    found_dir = None
    for path in all_input_paths:
        if os.path.isdir(path) and os.path.basename(path).lower() == folder_name.lower():
            found_dir = path
            break
    if not found_dir:
        for path in all_input_paths:
            if os.path.isdir(path):
                base = os.path.basename(path).lower()
                if any(p in base for p in patterns):
                    files = os.listdir(path)
                    if any(f.endswith(('.inter', '.pkl', '.npy')) for f in files):
                        found_dir = path
                        break
    if not found_dir and os.path.exists(os.path.join(CANONICAL_DATA_ROOT, folder_name)):
        found_dir = os.path.join(CANONICAL_DATA_ROOT, folder_name)

    if found_dir:
        safe_stage_dataset(found_dir, folder_name)
        prepared_data[key] = os.path.join(CANONICAL_DATA_ROOT, folder_name)
        file_stats = {f: os.path.getsize(os.path.join(found_dir, f)) for f in os.listdir(found_dir) if os.path.isfile(os.path.join(found_dir, f))}
        data_manifest[key] = {'source': found_dir, 'files': file_stats}
        print(f"  * {key.upper():12s} [FOUND] -> {found_dir}")
        print(f"    └──> Staged sang: {os.path.join(CANONICAL_DATA_ROOT, folder_name)}")
    else:
        print(f"  * {key.upper():12s} [MISSING] -> Chưa tìm thấy dữ liệu trong /kaggle/input.")

# Lưu data manifest để phục vụ kiểm chứng
manifest_path = os.path.join('/kaggle/working', 'data_manifest.json') if os.path.exists('/kaggle/working') else 'data_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(data_manifest, f, indent=2)

print("=" * 80)
print(f"✅ Đã chuẩn bị xong {len(prepared_data)}/{len(TARGET_DATASETS)} tập dữ liệu: {list(prepared_data.keys())}")

## Cell 3: STAIR4-v2.1 BCCR pre-flight suite

This cell compiles the implementation and runs `tests/test_stair4_v2_1.py` in a clean process. The suite covers the corrected Gate 0 FSC normalization, bounded encoders, stop-gradient, multi-positive supervision, optimizer isolation, and exact BSC recovery at `xi=0`.


In [ ]:
# Cell 3: STAIR4-v2.1 pre-flight contract suite
import os, py_compile, subprocess, sys

for path in ['/kaggle/working/STAIR-Enhanced', os.path.abspath('.')]:
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

module_files = [
    'models/freerec_compat.py', 'models/stair4_v2_utils.py',
    'models/stair4_heads.py', 'optimizers/stair4_v2_smoother.py',
    'models/stair4_v2.py', 'main_stair4_v2.py',
]
for module_file in module_files:
    py_compile.compile(module_file, doraise=True)
    print(f'PASS syntax: {module_file}')

command = [sys.executable, 'tests/test_stair4_v2_1.py']
print('Running:', ' '.join(command))
completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
if completed.returncode:
    raise RuntimeError('STAIR4-v2.1 pre-flight suite failed; do not start training.')
print('PASS: STAIR4-v2.1 Gate 0/BCCR/smoother contract suite')



## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU VRAM Profiler & Log Parsers
Khởi tạo động cơ giám sát thực nghiệm theo chuẩn công bố khoa học (Publication Standard):
- **Phân biệt rõ ràng Telemetry:**
  - `pynvml`: Theo dõi bộ nhớ tổng thể của toàn GPU thiết bị (Host GPU Allocated Memory).
  - PyTorch Native: Trích xuất chính xác `torch.cuda.max_memory_allocated()` và `max_memory_reserved()` thuần của mô hình từ tệp log.
- **Strict Metric Extractor:** Sửa lỗi P0, hỗ trợ cả hai dạng log FreeRec (`@Epoch 365` và `@Epoch: 365`).
- **An toàn Resume:** Hỗ trợ chế độ append (`mode='a'`) khi tiếp tục phiên chạy để tránh ghi đè lịch sử loss.
- **Trực quan hóa Trung thực:** Không hiển thị số liệu giả định hay đường cong vẽ sẵn khi chưa có dữ liệu đo lường thật.

In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

# Mốc đối chứng thực nghiệm chính thức từ các giai đoạn trước
BASELINE_REF = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0663, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V5_REF = {
    'baby':        {'Recall@10': 0.0669, 'Recall@20': 0.1027, 'NDCG@10': 0.0362, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0753, 'Recall@20': 0.1113, 'NDCG@10': 0.0415, 'NDCG@20': 0.0508},
    'electronics': {'Recall@10': 0.0451, 'Recall@20': 0.0678, 'NDCG@10': 0.0252, 'NDCG@20': 0.0311},
}

MHD_V3_REF = {
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1030, 'NDCG@10': 0.0362, 'NDCG@20': 0.0452},
    'sports':      {'Recall@10': 0.0758, 'Recall@20': 0.1122, 'NDCG@10': 0.0418, 'NDCG@20': 0.0512},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0685, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317},
}

DATASET_PROFILES = {
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'sparsity':    '99.88%',
        'color':       '#1f77b4',
        'approx_mins': 22.0,
    },
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'sparsity':    '99.95%',
        'color':       '#ff7f0e',
        'approx_mins': 52.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.69M Interactions',
        'sparsity':    '99.986%',
        'color':       '#2ca02c',
        'approx_mins': 310.0,
    },
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    '''Luồng nền theo dõi bộ nhớ GPU toàn hệ thống (Host GPU Allocated via NVML).'''
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def resolve_log_path(log_path):
    if os.path.exists(log_path):
        return log_path
    base = os.path.basename(log_path)
    candidates = [
        log_path,
        os.path.join('/kaggle/working/logs/stair4_v2', base),
        os.path.join('logs/stair4_v2', base),
        os.path.join('../logs/stair4_v2', base),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return log_path

def extract_test_metrics(log_path):
    '''Trích xuất chính xác TEST metrics CHỈ SAU KHI nạp checkpoint tốt nhất (Đã sửa P0 regex).'''
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # 1. Từ chối nếu có Traceback báo lỗi
    if 'Traceback (most recent call last):' in content[-2000:]:
        return None, {}

    # 2. Tìm dấu vết Load best model của FreeRec (Hỗ trợ cả '@Epoch 365' và '@Epoch: 365')
    best_match = re.search(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch\s*:?\s*(\d+)', content)
    if not best_match:
        return None, {}

    best_model_epoch = int(best_match.group(1))
    content_after_load = content[best_match.end():]

    # 3. Tìm khối TEST chính thức
    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after_load, re.DOTALL)
    if not test_match:
        return None, {}

    test_epoch = int(test_match.group(1))
    if test_epoch != best_model_epoch:
        return None, {}

    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))

    if len(test_metrics) < len(TRACKED_METRICS):
        return None, {}
    return test_epoch, test_metrics

def extract_gpu_telemetry(log_path):
    '''Trích xuất telemetry bộ nhớ tensor thuần của PyTorch từ tệp log.'''
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    alloc_m = re.search(r'Peak allocated\s*:\s*([0-9.]+)\s*MB', content)
    res_m = re.search(r'Peak reserved\s*:\s*([0-9.]+)\s*MB', content)
    telemetry = {}
    if alloc_m:
        telemetry['peak_allocated_mb'] = float(alloc_m.group(1))
    if res_m:
        telemetry['peak_reserved_mb'] = float(res_m.group(1))
    return telemetry

def parse_training_losses(log_path):
    '''Trích xuất quỹ đạo loss BPR và Total Loss.'''
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return [], []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'TRAIN @Epoch:\s*(\d+).*?LOSS\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    total_losses = [(int(ep), float(loss)) for ep, loss in matches]
    return total_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_log_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair4_v2(key, config_yaml, data_root, log_path, **kwargs):
    """Run a STAIR4-v2.1 BCCR experiment with documented CLI overrides only."""
    cfg_dict = STAIR4_V2_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)
    is_resuming = bool(cfg_dict.get('resume', False))

    if not os.path.exists(config_yaml):
        candidate = os.path.join('configs', os.path.basename(config_yaml))
        if os.path.exists(candidate):
            config_yaml = candidate
    if not os.path.exists(config_yaml):
        raise FileNotFoundError(f'Configuration file not found: {config_yaml}')

    print('=' * 80)
    print(f'RUNNING STAIR4-v2.1 (BCCR): {key.upper()}')
    for label, setting in [
        ('YAML', config_yaml),
        ('Auxiliary kernel', cfg_dict.get('auxiliary_kernel')),
        ('Rotation', cfg_dict.get('rotation_mode')),
        ('Smoother', cfg_dict.get('smoother_mode')),
        ('lambda_max', cfg_dict.get('pocl_weight_target')),
        ('xi', cfg_dict.get('xi')),
        ('Ablation', cfg_dict.get('ablation_id')),
    ]:
        print(f'  {label:18s}: {setting}')
    print('=' * 80)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair4_v2.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair4_v2.py'

    command = [sys.executable, runner_py, '--config', config_yaml, '--root', data_root]
    supported = [
        'seed', 'device', 'num_workers', 'id', 'resume',
        'auxiliary_kernel', 'rotation_mode', 'smoother_mode', 'eta', 'kappa',
        'pocl_weight_target', 'contrastive_temperature', 'warmup_epochs', 'ramp_epochs',
        'xi', 'aux_lr_ratio', 'aux_weight_decay', 'knn_block_size', 'query_chunk_size',
        'ablation_id', 'batch_size', 'epochs', 'lr', 'weight_decay',
    ]
    for name in supported:
        value = cfg_dict.get(name)
        if value is None:
            continue
        flag = '--' + name.replace('_', '-')
        if isinstance(value, bool):
            if value:
                command.append(flag)
        else:
            command.extend([flag, str(value)])

    print('Launching:', ' '.join(command))
    stop_evt = threading.Event()
    monitor = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    monitor.start()
    started = time.time()
    try:
        with open(log_path, 'a' if is_resuming else 'w', encoding='utf-8') as stream:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1, env={**os.environ, 'PYTHONWARNINGS': 'ignore::FutureWarning'})
            for line in process.stdout:
                print(line, end='')
                stream.write(line)
            process.wait()
        if process.returncode:
            raise RuntimeError(f'Training failed with exit code {process.returncode}; inspect {log_path}')
    finally:
        stop_evt.set()
        monitor.join(timeout=3.0)

    print(f'Completed in {(time.time() - started) / 60.0:.2f} minutes.')
    epoch, metrics = extract_test_metrics(log_path)
    if metrics:
        print(f'Official test metrics at best epoch {epoch}:')
        for metric in TRACKED_METRICS:
            print(f'  {metric}: {metrics[metric]:.4f}')
    return epoch, metrics

def plot_vram_profile(key, dataset_name=None, output_filename=None):
    '''Trực quan hóa mức tiêu thụ bộ nhớ GPU (NVML Telemetry). Không dùng số liệu giả định.'''
    import matplotlib.pyplot as plt
    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#1f77b4'})
    disp_name = dataset_name if dataset_name else info['name']
    color = info['color']

    fig, ax = plt.subplots(figsize=(10, 4.2), dpi=150)
    records = vram_profile.get(key, [])

    if records:
        ts = np.arange(len(records)) * 2.0 / 60.0  # phút
        ax.plot(ts, records, color=color, lw=1.8, label=f'{disp_name} Measured Host GPU (NVML)')
        peak = max(records)
        ax.axhline(peak, color='#d62728', linestyle='--', lw=1.2, label=f'Peak Host VRAM: {peak:.1f} MiB')
        ax.set_xlabel('Thời gian huấn luyện (phút)', fontsize=10.5)
        ax.set_title(f'Host GPU Memory Utilization (NVML) — {disp_name}', fontsize=12.5, fontweight='bold')
        ax.legend(loc='lower right', fontsize=9.0)
    else:
        ax.text(0.5, 0.5, "Chưa có bản ghi GPU telemetry thực tế\n(Biểu đồ sẽ được hiển thị sau khi thực thi cell huấn luyện)",
                ha='center', va='center', transform=ax.transAxes, color='#777777', fontsize=11)
        ax.set_title(f'Host GPU Memory Utilization — {disp_name} [Đang chờ thực thi]', fontsize=12.5, fontweight='bold')

    ax.set_ylabel('Host GPU Memory (MiB)', fontsize=10.5)
    ax.grid(True, linestyle='--', alpha=0.35)
    plt.tight_layout()
    if not output_filename:
        output_filename = f'/kaggle/working/reports/vram_profile_{key}.png'
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()

def plot_single_dataset_learning_curves(key, dataset_name=None, output_filename=None):
    '''Đồ thị 4 panel chuẩn mực theo dõi động lực học học tập STAIR4-v2.'''
    import matplotlib.pyplot as plt
    info = DATASET_PROFILES.get(key, {'name': key.capitalize(), 'color': '#1f77b4'})
    disp_name = dataset_name if dataset_name else info['name']

    cfg_item = STAIR4_V2_CONFIGS.get(key, {})
    log_file = cfg_item.get('log', f'{key}_stair4_v2.log')

    _, total_losses = parse_training_losses(log_file)
    val_ndcg = parse_valid_metric(log_file, 'NDCG@20')
    val_recall = parse_valid_metric(log_file, 'Recall@20')

    preview_mode = (len(total_losses) == 0)

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), dpi=150)
    title_suffix = ' [Chờ Chạy Huấn Luyện]' if preview_mode else ''
    fig.suptitle(f'STAIR4-v2.1 BCCR Training and Validation Dynamics — {disp_name}{title_suffix}',
                 fontsize=14, fontweight='bold', y=0.98)

    # 1. Training Loss
    ax_loss = axes[0]
    if not preview_mode and total_losses:
        eps_t, l_tot = zip(*total_losses)
        ax_loss.plot(eps_t, l_tot, color='#1f77b4', lw=1.8, label='Total Loss (BPR + BCCR)')
        ax_loss.set_xlim(1, max(eps_t[-1], 2))
        ax_loss.legend(loc='upper right', fontsize=8.5)
    else:
        ax_loss.text(0.5, 0.5, "Chưa có dữ liệu loss\n(Thực thi cell huấn luyện để ghi nhận)",
                     ha='center', va='center', transform=ax_loss.transAxes, color='#777777', fontsize=10.5)
    ax_loss.set_title('(a) Training Loss Trajectory', fontweight='bold', fontsize=11.5)
    ax_loss.set_xlabel('Epoch', fontsize=10)
    ax_loss.set_ylabel('Loss Value', fontsize=10)
    ax_loss.grid(True, linestyle='--', alpha=0.35)

    # 2. Validation NDCG@20
    ax_ndcg = axes[1]
    best_n_ep, best_n_val = 0, 0.0
    if not preview_mode and val_ndcg:
        eps_n, ndcgs = zip(*val_ndcg)
        best_n_ep, best_n_val = max(val_ndcg, key=lambda x: x[1])
        ax_ndcg.plot(eps_n, ndcgs, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
        ax_ndcg.axvline(best_n_ep, color='#777777', linestyle='--', lw=1.2, alpha=0.75, label=f'Best ({best_n_ep})')
        ax_ndcg.legend(loc='lower right', fontsize=8.5)
    else:
        ax_ndcg.text(0.5, 0.5, "Chưa có validation NDCG@20\n(Đang chờ huấn luyện)",
                     ha='center', va='center', transform=ax_ndcg.transAxes, color='#777777', fontsize=10.5)
    ax_ndcg.set_title('(b) Validation NDCG@20', fontweight='bold', fontsize=11.5)
    ax_ndcg.set_xlabel('Epoch', fontsize=10)
    ax_ndcg.set_ylabel('Score', fontsize=10)
    ax_ndcg.grid(True, linestyle='--', alpha=0.35)

    # 3. Validation Recall@20
    ax_rec = axes[2]
    if not preview_mode and val_recall:
        eps_r, recalls = zip(*val_recall)
        best_r_ep, best_r_val = max(val_recall, key=lambda x: x[1])
        ax_rec.plot(eps_r, recalls, color='#9467bd', lw=2.0, label='Validation Recall@20')
        ax_rec.axvline(best_r_ep, color='#777777', linestyle='--', lw=1.2, alpha=0.75, label=f'Best ({best_r_ep})')
        ax_rec.legend(loc='lower right', fontsize=8.5)
    else:
        ax_rec.text(0.5, 0.5, "Chưa có validation Recall@20\n(Đang chờ huấn luyện)",
                     ha='center', va='center', transform=ax_rec.transAxes, color='#777777', fontsize=10.5)
    ax_rec.set_title('(c) Validation Recall@20', fontweight='bold', fontsize=11.5)
    ax_rec.set_xlabel('Epoch', fontsize=10)
    ax_rec.set_ylabel('Score', fontsize=10)
    ax_rec.grid(True, linestyle='--', alpha=0.35)

    # 4. Schedule dynamics (BCCR lambda; xi remains zero for baseline recovery)
    ax_sch = axes[3]
    eps_s = np.arange(1, 101)
    w, r = 10, 20
    lam_target, xi_target = 0.0001, 0.0
    lam_v = np.clip((eps_s - w) / float(r), 0.0, 1.0) * lam_target
    xi_v = np.clip((eps_s - w) / float(r), 0.0, 1.0) * xi_target
    ax_sch.plot(eps_s, lam_v, color='#d62728', lw=1.8, label='BCCR Weight (λ_cl)')
    ax_z = ax_sch.twinx()
    ax_z.plot(eps_s, xi_v, color='#17becf', linestyle='--', lw=2.0, label='Residual correction (ξ)')
    ax_z.set_ylabel('ξ Value', color='#17becf', fontsize=10)
    ax_sch.set_title('(d) BCCR and residual-smoother schedules (λ & ξ)', fontweight='bold', fontsize=11.5)
    ax_sch.set_xlabel('Epoch', fontsize=10)
    ax_sch.set_ylabel('λ_cl Value', color='#d62728', fontsize=10)
    lines1, labels1 = ax_sch.get_legend_handles_labels()
    lines2, labels2 = ax_z.get_legend_handles_labels()
    ax_sch.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8.0)
    ax_sch.grid(True, linestyle='--', alpha=0.35)

    plt.tight_layout(rect=[0, 0.045, 1, 0.96])
    if not output_filename:
        output_filename = f'/kaggle/working/reports/learning_curve_{key}.png'
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ [Đồ thị hoàn tất] -> {output_filename}')


## Cell 5 📋 Cấu hình Siêu tham số STAIR4-v2 (Dataset-Adaptive Matrix)
Cấu hình dựa trên đặc tả tại `STAIR4_v2_Report.md` (§6 & §14) và kế thừa từ baseline YAML:
- **Thứ tự thực nghiệm:** **Amazon Baby** $\rightarrow$ **Amazon Sports** $\rightarrow$ **Amazon Electronics**.
- **Chế độ Chạy:**
  - `single`: Chạy một biến thể đại diện (Mặc định `A6`: Full BSF + POCL) đối chứng với các mô hình baseline lịch sử.
  - `ablation_sweep`: Cho phép chạy quét toàn bộ ma trận ablation `['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7']` dưới cùng một protocol.

In [ ]:
# Cell 5: STAIR4-v2.1 BCCR configuration matrix
import os

os.makedirs('/kaggle/working/logs/stair4_v2', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)

# C0 is the baseline-recovery control.  C2 is the recommended primary v2.1 pilot.
SELECTED_ABLATION = 'C2'
ABLATION_PRESETS = {
    'C0': {'auxiliary_kernel': 'none',   'rotation_mode': 'identity',       'smoother_mode': 'baseline',          'xi': 0.0},
    'C1': {'auxiliary_kernel': 'cosine', 'rotation_mode': 'identity',       'smoother_mode': 'baseline',          'xi': 0.0},
    'C2': {'auxiliary_kernel': 'hybrid', 'rotation_mode': 'identity',       'smoother_mode': 'baseline',          'xi': 0.0},
    'C3': {'auxiliary_kernel': 'hybrid', 'rotation_mode': 'learned_givens', 'smoother_mode': 'baseline',          'xi': 0.0},
    'C4': {'auxiliary_kernel': 'hybrid', 'rotation_mode': 'identity',       'smoother_mode': 'residual_spectral', 'xi': 0.02},
}
if SELECTED_ABLATION not in ABLATION_PRESETS:
    raise ValueError(f'Unknown v2.1 ablation: {SELECTED_ABLATION}')

selected = ABLATION_PRESETS[SELECTED_ABLATION]
common = {
    **selected, 'ablation_id': SELECTED_ABLATION,
    'eta': 0.25, 'kappa': 0.5, 'contrastive_temperature': 0.2,
    'pocl_weight_target': 0.0001, 'warmup_epochs': 10, 'ramp_epochs': 20,
    'aux_lr_ratio': 0.1, 'aux_weight_decay': 0.0,
    'knn_block_size': 256, 'query_chunk_size': 256,
}

STAIR4_V2_CONFIGS = {
    'baby': {**common,
        'yaml': '/kaggle/working/STAIR-Enhanced/configs/dataset_stair4_v2_baby.yaml',
        'log': '/kaggle/working/logs/stair4_v2/baby_stair4_v21.log'},
    'sports': {**common,
        'yaml': '/kaggle/working/STAIR-Enhanced/configs/dataset_stair4_v2_sports.yaml',
        'log': '/kaggle/working/logs/stair4_v2/sports_stair4_v21.log'},
    'electronics': {**common,
        'yaml': '/kaggle/working/STAIR-Enhanced/configs/dataset_stair4_v2_electronics.yaml',
        'log': '/kaggle/working/logs/stair4_v2/electronics_stair4_v21.log',
        'query_chunk_size': 128},
}
print(f'Selected STAIR4-v2.1 ablation: {SELECTED_ABLATION} -> {selected}')



## Train STAIR4-v2.1 on Amazon Baby

Run the recommended **C2** BCCR pilot first. It preserves the baseline BSC smoother (`xi=0`) and activates the train-only auxiliary loss only after the warm-up/ramp schedule.


In [ ]:
# STAIR4-v2.1 BCCR training: Amazon Baby
DATA_ROOT = CANONICAL_DATA_ROOT
config = STAIR4_V2_CONFIGS['baby']
if 'baby' not in prepared_data:
    raise RuntimeError('Dataset preparation is incomplete; run the data-preparation cell first.')
run_training_stair4_v2(
    key='baby', config_yaml=config['yaml'], data_root=DATA_ROOT, log_path=config['log'],
)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ VRAM (NVML) — Amazon Baby
Đo lường chi phí bộ nhớ GPU thực tế trên tập Amazon Baby.

In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Baby
plot_vram_profile('baby')

## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby
Hiển thị 4 panel độc lập: Loss Trajectory, Validation NDCG@20, Validation Recall@20 và Schedule Dynamics.

In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Baby
plot_single_dataset_learning_curves('baby')

## Train STAIR4-v2.1 on Amazon Sports

Use the same selected v2.1 ablation and compare the loaded-best-checkpoint test metrics with the baseline reference.


In [ ]:
# STAIR4-v2.1 BCCR training: Amazon Sports
DATA_ROOT = CANONICAL_DATA_ROOT
config = STAIR4_V2_CONFIGS['sports']
if 'sports' not in prepared_data:
    raise RuntimeError('Dataset preparation is incomplete; run the data-preparation cell first.')
run_training_stair4_v2(
    key='sports', config_yaml=config['yaml'], data_root=DATA_ROOT, log_path=config['log'],
)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ VRAM (NVML) — Amazon Sports
Đo lường chi phí bộ nhớ GPU thực tế trên tập Amazon Sports.

In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Sports
plot_vram_profile('sports')

## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports
Hiển thị 4 panel độc lập theo dõi tiến trình xếp hạng và mất mát của Amazon Sports.

In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Sports
plot_single_dataset_learning_curves('sports')

## Train STAIR4-v2.1 on Amazon Electronics

The Electronics configuration uses a smaller contrastive query chunk (`128`) to cap temporary BCCR logits memory. `xi=0` remains the safe baseline-recovery default.


In [ ]:
# STAIR4-v2.1 BCCR training: Amazon Electronics
DATA_ROOT = CANONICAL_DATA_ROOT
config = STAIR4_V2_CONFIGS['electronics']
if 'electronics' not in prepared_data:
    raise RuntimeError('Dataset preparation is incomplete; run the data-preparation cell first.')
run_training_stair4_v2(
    key='electronics', config_yaml=config['yaml'], data_root=DATA_ROOT, log_path=config['log'],
)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ VRAM (NVML) — Amazon Electronics
Đo lường khả năng kiểm soát bộ nhớ VRAM an toàn trên không gian 63K items.

In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics
plot_vram_profile('electronics')

## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics
Hiển thị 4 panel động lực học trên tập Amazon Electronics.

In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
plot_single_dataset_learning_curves('electronics')

## Cell 9 📊 Bảng So sánh Tham chiếu Đa Mô hình (Cross-Model Reference Benchmark — 4 Chỉ số)
Đối chiếu toàn diện 4 chỉ số chuẩn: **Recall@10, Recall@20, NDCG@10, NDCG@20** giữa:
1. STAIR Baseline (Chuẩn MMRec)
2. STAIR-BSC-Reweight (v5 Giai đoạn 2)
3. STAIR-MHD v3 (Giai đoạn 3-v3 / Giai đoạn 4)
4. **STAIR4-v2 (BSF–POCL: Bounded Spectral Filtering & Phase-Overlap Contrastive)**

*Nguyên tắc liêm chính học thuật:* Chỉ những tập dữ liệu đã hoàn tất và có kết quả kiểm thử chính thức (`TEST @Epoch`) mới được ghi nhận chỉ số thực nghiệm. Dataset chưa chạy xong hiển thị `N/A`.

In [ ]:
# Cell 9: Bảng so sánh Tham chiếu Đa Mô hình (Recall@10, Recall@20, NDCG@10, NDCG@20)
import os
try:
    from prettytable import PrettyTable
    USE_PRETTYTABLE = True
except ImportError:
    USE_PRETTYTABLE = False

# Thứ tự hiển thị bảng: Baby -> Sports -> Electronics
ORDERED_KEYS = ['baby', 'sports', 'electronics']

headers = ['Dataset', 'Model Architecture', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20', 'Trạng thái']
rows = []

for key in ORDERED_KEYS:
    dname = DATASET_PROFILES[key]['name']

    # 1. Baseline
    b = BASELINE_REF.get(key, {})
    rows.append([dname, 'STAIR (Baseline)', f"{b.get('Recall@10', 0):.4f}", f"{b.get('Recall@20', 0):.4f}",
                 f"{b.get('NDCG@10', 0):.4f}", f"{b.get('NDCG@20', 0):.4f}", 'Official Ref'])

    # 2. V5
    v5 = V5_REF.get(key, {})
    rows.append([dname, 'STAIR-BSC-Reweight (v5)', f"{v5.get('Recall@10', 0):.4f}", f"{v5.get('Recall@20', 0):.4f}",
                 f"{v5.get('NDCG@10', 0):.4f}", f"{v5.get('NDCG@20', 0):.4f}", 'Official Ref'])

    # 3. MHD v3
    v3 = MHD_V3_REF.get(key, {})
    rows.append([dname, 'STAIR-MHD v3', f"{v3.get('Recall@10', 0):.4f}", f"{v3.get('Recall@20', 0):.4f}",
                 f"{v3.get('NDCG@10', 0):.4f}", f"{v3.get('NDCG@20', 0):.4f}", 'Official Ref'])

    # 4. STAIR4-v2 (BSF-POCL)
    log_p = STAIR4_V2_CONFIGS[key]['log']
    t_ep, t_res = extract_test_metrics(log_p)
    if t_res:
        r10 = f"{t_res.get('Recall@10', 0):.4f}"
        r20 = f"{t_res.get('Recall@20', 0):.4f}"
        n10 = f"{t_res.get('NDCG@10', 0):.4f}"
        n20 = f"{t_res.get('NDCG@20', 0):.4f}"
        status = f"✅ Measured (@Ep {t_ep})"
    else:
        r10 = r20 = n10 = n20 = 'N/A'
        status = '⏳ Chưa chạy / Đang chạy'

    rows.append([dname, f'STAIR4-v2.1 ({SELECTED_ABLATION})', r10, r20, n10, n20, status])

if USE_PRETTYTABLE:
    table = PrettyTable()
    table.field_names = headers
    for r in rows:
        table.add_row(r)
    print(table)
else:
    col_w = [18, 25, 11, 11, 11, 11, 24]
    print(' | '.join(h.center(col_w[i]) for i, h in enumerate(headers)))
    print('-' * (sum(col_w) + len(col_w) * 3))
    for r in rows:
        print(' | '.join(str(val).center(col_w[i]) for i, val in enumerate(r)))

## Cell 10: Multi-dataset convergence visualisation

The publication plots compare training loss, validation NDCG@20, validation Recall@20, and the configured BCCR weight schedule. The residual correction remains `xi=0` for the baseline-recovery pilot.


In [ ]:
# Cell 10: Learning Dynamics & Multi-Dataset Convergence Trajectories (Publication Standard)
import os
import matplotlib.pyplot as plt
import numpy as np

active_keys = [k for k in ['baby', 'sports', 'electronics'] if k in STAIR4_V2_CONFIGS and os.path.exists(STAIR4_V2_CONFIGS[k]['log'])]
preview_mode = (len(active_keys) == 0)

if preview_mode:
    active_keys = ['baby', 'sports', 'electronics']
    print('ℹ️ Ghi chú: Hiển thị chế độ đồ thị tham chiếu đối chiếu mẫu.')

fig, axes = plt.subplots(len(active_keys), 3, figsize=(18, 4.5 * len(active_keys)), dpi=150)
if len(active_keys) == 1:
    axes = np.expand_dims(axes, 0)

for idx, key in enumerate(active_keys):
    info = DATASET_PROFILES[key]
    color = info['color']
    log_f = STAIR4_V2_CONFIGS[key]['log']

    _, total_losses = parse_training_losses(log_f)
    val_ndcg = parse_valid_metric(log_f, 'NDCG@20')

    # Panel 1: Loss
    ax_l = axes[idx, 0]
    if total_losses:
        eps, l_val = zip(*total_losses)
        ax_l.plot(eps, l_val, color=color, lw=1.8, label='Total Loss')
        ax_l.set_xlim(1, max(eps[-1], 2))
    else:
        ax_l.text(0.5, 0.5, f"Chưa có log {info['name']}", ha='center', va='center', transform=ax_l.transAxes, color='#777')
    ax_l.set_title(f"{info['name']} — Training Loss", fontweight='bold')
    ax_l.set_xlabel('Epoch')
    ax_l.set_ylabel('Loss')
    ax_l.grid(True, linestyle='--', alpha=0.35)

    # Panel 2: Validation NDCG@20
    ax_n = axes[idx, 1]
    if val_ndcg:
        eps_n, ndcg_vals = zip(*val_ndcg)
        ax_n.plot(eps_n, ndcg_vals, color='#2ca02c', lw=2.0, label='Validation NDCG@20')
        b_ep, b_val = max(val_ndcg, key=lambda x: x[1])
        ax_n.axvline(b_ep, color='#777', linestyle='--', lw=1.2, label=f'Best ({b_ep})')
        ax_n.legend(loc='lower right', fontsize=8)
    else:
        ax_n.text(0.5, 0.5, f"Chưa có validation {info['name']}", ha='center', va='center', transform=ax_n.transAxes, color='#777')
    ax_n.set_title(f"{info['name']} — Validation NDCG@20", fontweight='bold')
    ax_n.set_xlabel('Epoch')
    ax_n.set_ylabel('NDCG@20')
    ax_n.grid(True, linestyle='--', alpha=0.35)

    # Panel 3: Schedule Dynamics
    ax_s = axes[idx, 2]
    eps_s = np.arange(1, 101)
    lam = np.clip((eps_s - 10) / 20.0, 0.0, 1.0) * 0.001
    zet = np.clip((eps_s - 10) / 20.0, 0.0, 1.0) * 0.10
    ax_s.plot(eps_s, lam, color='#d62728', lw=1.8, label='λ_cl (POCL)')
    ax_sz = ax_s.twinx()
    ax_sz.plot(eps_s, zet, color='#17becf', linestyle='--', lw=2.0, label='ζ (BSF)')
    ax_sz.set_ylabel('ζ Value', color='#17becf')
    ax_s.set_title(f"{info['name']} — Regularization Schedule", fontweight='bold')
    ax_s.set_xlabel('Epoch')
    ax_s.set_ylabel('λ_cl Value', color='#d62728')
    ax_s.grid(True, linestyle='--', alpha=0.35)

plt.tight_layout()
out_fig = '/kaggle/working/reports/stair4_v2_multi_dataset_convergence.png'
os.makedirs(os.path.dirname(out_fig), exist_ok=True)
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Đồ thị hội tụ đa tập dữ liệu đã lưu] -> {out_fig}')

## Cell 10b 🔬 Figure 4 — Phân Tích Phổ Toán Tử Bounded Spectral Filtering (BSF)
Trực quan hóa giải tích toán học hàm truyền phổ của bộ lọc $Q_t(x) = x - \frac{t^2}{2} H(H(x))$ với $H = \frac{I - S}{2}$:
$$p_t(\lambda) = 1 - \frac{t^2}{8}(1 - \lambda)^2, \quad \lambda \in [-1, 1]$$
- Tại $\lambda = 1$ (thành phần DC): $p_t(1) = 1.0$ (bảo toàn hoàn hảo tín hiệu đồ thị).
- Tại $\lambda = -1$ (tần số cao nhất): $p_t(-1) = 1 - \frac{t^2}{2} \ge 0.5$ với $t \in [0, 1]$ (chặn biên dưới chặt, triệt tiêu nhiễu tần số cao mà không làm suy biến không gian embedding).

In [ ]:
# Cell 10b: Figure 4 — BSF Spectral Transfer Function & Eigenvalue Shrinkage [Analytical Formula]
import matplotlib.pyplot as plt
import numpy as np

lambdas = np.linspace(-1.0, 1.0, 500)
t_values = [0.0, 0.25, 0.50, 0.75, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), dpi=150)
fig.suptitle('Figure 4: Bounded Spectral Filtering (BSF) Operator Spectral Transfer Analysis [Analytical Formula]',
             fontsize=13, fontweight='bold', y=0.98)

# Panel A: Spectral Transfer Function p_t(lambda)
ax0 = axes[0]
for t in t_values:
    # p_t(lambda) = 1 - (t^2 / 8) * (1 - lambda)^2
    p_lambda = 1.0 - (t**2 / 8.0) * ((1.0 - lambdas)**2)
    style = '-' if t in [0.0, 0.5, 1.0] else '--'
    lw = 2.2 if t == 0.5 else 1.6
    ax0.plot(lambdas, p_lambda, style, lw=lw, label=f't = {t:.2f} ({"Pilot" if t==0.5 else ("Identity" if t==0 else "Max")})')

ax0.axhline(0.5, color='#d62728', linestyle=':', lw=1.2, label='Lower Bound (p >= 0.5 at t=1)')
ax0.set_title('(a) Spectral Transfer Curve p_t(λ) [Exact Formula]', fontweight='bold')
ax0.set_xlabel('Item Graph Eigenvalue λ ∈ [-1, 1]')
ax0.set_ylabel('Transfer Multiplier p_t(λ)')
ax0.set_ylim(0.40, 1.05)
ax0.grid(True, linestyle='--', alpha=0.35)
ax0.legend(loc='lower left', fontsize=9.0)

# Panel B: Attenuation Ratio vs Spectral Time t
ax1 = axes[1]
ts = np.linspace(0.0, 1.0, 200)
high_freq_mult = 1.0 - 0.5 * (ts**2)
med_freq_mult = 1.0 - 0.125 * (ts**2)

ax1.plot(ts, high_freq_mult, color='#d62728', lw=2.0, label='High Frequency (λ = -1)')
ax1.plot(ts, med_freq_mult, color='#1f77b4', lw=2.0, label='Mid Frequency (λ = 0)')
ax1.axvline(0.5, color='#777', linestyle='--', lw=1.2, label='Pilot Setting (t = 0.5)')
ax1.set_title('(b) High-Frequency Attenuation vs Spectral Time t', fontweight='bold')
ax1.set_xlabel('Spectral Time t ∈ [0, 1]')
ax1.set_ylabel('Effective Multiplier')
ax1.set_ylim(0.45, 1.02)
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.legend(loc='lower left', fontsize=9.0)

plt.tight_layout()
out_f4 = '/kaggle/working/reports/figure4_bsf_spectral_transfer.png'
plt.savefig(out_f4, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Figure 4 đã lưu] -> {out_f4}')

## Cell 10c 🔬 Figure 5 — Động Lực Học Coherent Phase-Overlap & Phép Quay Givens
Trực quan hóa hình học không gian pha phức:
- So sánh hàm mất mát giữa **Phase Fidelity** $\mathcal{F}(\psi_u, \psi_i) = |\langle \psi_u, \psi_i \rangle|^2 \in [0, 1]$ và **Cosine Similarity** $\in [-1, 1]$.
- Phân tích động lực học góc xoay Givens $\theta_k \in \mathbb{R}^{d/2}$ trên $d/2$ mặt phẳng con 2D độc lập.

*(Ghi chú: Panel (a) là hàm hình học giải tích; Panel (b) là mô phỏng quỹ đạo trước thực nghiệm để minh họa cơ chế. Quỹ đạo góc xoay thực nghiệm chính thức sẽ được trích xuất từ checkpoint sau khi huấn luyện).*

In [ ]:
# Cell 10c: Figure 5 — Phase-Overlap Coherent Fidelity & Pairwise Givens Dynamics [Theoretical Analysis & Simulation]
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), dpi=150)
fig.suptitle('Figure 5: Signed/Fidelity Components and Pairwise Givens Dynamics [Illustrative]',
             fontsize=13, fontweight='bold', y=0.98)

# Panel A: Fidelity vs Inner Product Angle (Exact Formula)
delta_phi = np.linspace(-np.pi, np.pi, 500)
fidelity_1d = np.cos(delta_phi)**2
cosine_sim = np.cos(delta_phi)

ax0 = axes[0]
ax0.plot(delta_phi, fidelity_1d, color='#9467bd', lw=2.2, label='Fidelity component |⟨ψ_u, ψ_i⟩|²')
ax0.plot(delta_phi, cosine_sim, color='#777', linestyle='--', lw=1.6, label='Cosine Control (Real domain)')
ax0.set_title('(a) Geometric Response vs Phase Difference Δφ [Exact]', fontweight='bold')
ax0.set_xlabel('Phase Difference Δφ (radians)')
ax0.set_ylabel('Similarity Metric')
ax0.grid(True, linestyle='--', alpha=0.35)
ax0.legend(loc='lower center', fontsize=9.0)

# Panel B: Pairwise Givens Rotation Trajectory Simulation
ax1 = axes[1]
epochs = np.arange(1, 101)
np.random.seed(42)
for k in range(4):
    target_angle = (np.random.rand() - 0.5) * 0.4
    theta_traj = target_angle * (1.0 - np.exp(-epochs / 15.0)) + np.random.normal(0, 0.01, size=len(epochs))
    ax1.plot(epochs, theta_traj, lw=1.8, label=f'Givens Block θ_{k+1}')

ax1.axhline(0.0, color='#777', linestyle=':', lw=1.2, label='Identity Init (θ=0)')
ax1.set_title('(b) Pairwise Givens Angle Trajectories θ_k [Illustrative Simulation]', fontweight='bold')
ax1.set_xlabel('Training Epoch')
ax1.set_ylabel('Rotation Angle θ (radians)')
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.legend(loc='lower right', fontsize=8.5)

plt.tight_layout()
out_f5 = '/kaggle/working/reports/figure5_phase_fidelity_givens.png'
plt.savefig(out_f5, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Figure 5 đã lưu] -> {out_f5}')

## Cell 11 ⚡ Biểu đồ Tổng Hợp Bộ Nhớ GPU Mô Hình 3 Tập Dữ Liệu
Tổng hợp mức tiêu thụ bộ nhớ GPU trên cả 3 tập dữ liệu Amazon Baby, Amazon Sports, Amazon Electronics:
- Đo lường qua NVML Telemetry thực tế.
- Nếu chưa có phiên chạy hoàn tất, biểu đồ hiển thị trạng thái chờ `N/A`, không vẽ số liệu giả định.

In [ ]:
# Cell 11: Comprehensive Multi-Dataset GPU Memory Utilization Benchmark
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), dpi=150)
fig.suptitle('Figure 6: Multi-Dataset GPU Memory Utilization Benchmark (STAIR4-v2 Measured Telemetry)',
             fontsize=13.5, fontweight='bold', y=0.98)

target_keys = ['baby', 'sports', 'electronics']

for idx, key in enumerate(target_keys):
    ax = axes[idx]
    info = DATASET_PROFILES[key]
    color = info['color']
    records = vram_profile.get(key, [])

    if records:
        ts = np.arange(len(records)) * 2.0 / 60.0
        ax.plot(ts, records, color=color, lw=2.0, label=f'{info["name"]}')
        peak = max(records)
        ax.axhline(peak, color='#d62728', linestyle='--', lw=1.2, label=f'Peak: {peak:.1f} MiB')
        ax.set_xlabel('Thời gian (phút)')
        ax.set_ylim(0, peak * 1.35)
        ax.legend(loc='lower right', fontsize=8.5)
    else:
        ax.text(0.5, 0.5, f"Chưa có dữ liệu đo lường {info['name']}\n(N/A — Đang chờ thực thi huấn luyện)",
                ha='center', va='center', transform=ax.transAxes, color='#777', fontsize=11)
        ax.set_xlabel('Thời gian')

    ax.set_title(f"{info['name']} ({info['scale'].split('|')[1].strip()})", fontweight='bold')
    ax.set_ylabel('VRAM (MiB)')
    ax.grid(True, linestyle='--', alpha=0.35)

plt.tight_layout()
out_vram = '/kaggle/working/reports/stair4_v2_vram_benchmark.png'
plt.savefig(out_vram, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ [Biểu đồ VRAM benchmark đã lưu] -> {out_vram}')

## Cell 12 💾 Xuất Báo Cáo Kết Quả CSV & Đoạn Mã LaTeX Cho Khóa Luận Tốt Nghiệp
Lưu trữ tự động bảng kết quả tổng hợp ra file CSV và sinh mã bảng biểu LaTeX chuẩn mực để chèn trực tiếp vào báo cáo Khóa luận.

In [ ]:
# Cell 12: Xuất bảng kết quả CSV và mã LaTeX cho Khóa Luận Tốt Nghiệp
import csv, os

WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
OUT_CSV = os.path.join(WORK_DIR, 'reports', 'stair4_v2_ablation_summary.csv')
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    for r in rows:
        writer.writerow(r)

print(f'✅ Bảng kết quả tổng hợp đã được lưu trữ thành công tại: {OUT_CSV}')

# Sinh đoạn mã LaTeX chuẩn bị cho Khóa Luận
print('\n' + '=' * 80)
print('% MÃ LATEX BẢNG KẾT QUẢ THỰC NGHIỆM KHÓA LUẬN TỐT NGHIỆP')
print('=' * 80)
latex_code = [
    r'\begin{table*}[t]',
    r'\centering',
    r'\caption{Hiệu năng xếp hạng đa phương thái trên 3 tập dữ liệu của STAIR4-v2 (BSF--POCL) so với các mô hình đường cơ sở.}',
    r'\label{tab:stair4_v2_results}',
    r'\small',
    r'\begin{tabular}{llcccc}',
    r'\toprule',
    r'\textbf{Dataset} & \textbf{Model} & \textbf{Recall@10} & \textbf{Recall@20} & \textbf{NDCG@10} & \textbf{NDCG@20} \\',
    r'\midrule',
]

prev_d = None
for r in rows:
    d, m, r10, r20, n10, n20, _ = r
    if prev_d and d != prev_d:
        latex_code.append(r'\midrule')
    prev_d = d
    latex_code.append(f"{d} & {m} & {r10} & {r20} & {n10} & {n20} \\\\")

latex_code.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table*}',
])

print('\n'.join(latex_code))
print('=' * 80)

## Thesis defense notes

The v2.1 claim should remain empirical: BCCR is train-only multi-positive regularisation with a stop-gradient key branch, while the default `xi=0` preserves the original BSC smoother exactly. Report the C0 recovery control and C1–C4 ablations; do not claim an improvement before the matched runs complete.
